# Pix2Pix Training Notebook

This notebook runs the full GAN training loop for the furniture placement project.

**Architecture recap:**
- **Generator (U-Net):** Takes a room image + furniture image → outputs a furnished room
- **Discriminator (PatchGAN):** Judges whether a (room, furnished-room) pair is real or fake

**How to use:**
1. Connect to a T4 GPU runtime via VSCode's Colab extension
2. Run Cell 1 to verify the GPU is detected
3. Run all cells in order
4. Optionally override values in Cell 5 before starting training

## Cell 1: Setup Data Access

**Option A (recommended):** Use Command Palette → `Colab: Mount Google Drive to Server...` to mount Drive, then run Cell 2.

**Option B:** Right-click `processed-256` folder in VSCode Explorer → "Upload to Colab". Data will appear at `/content/processed-256/`. Then run Cell 2.

In [ ]:
import os, shutil

# === Choose ONE of these paths depending on how you uploaded the data ===

# Option A: If you mounted Google Drive via Command Palette
# DATA_SOURCE = "/content/drive/MyDrive/processed-256"

# Option B: If you uploaded via right-click "Upload to Colab"
DATA_SOURCE = "/content/processed-256"

# -----------------------------------------------------------------
# Copy data to local disk for faster I/O during training
# -----------------------------------------------------------------
PROJECT_ROOT = "/content/project"

if not os.path.exists(os.path.join(PROJECT_ROOT, "data", "processed", "train")):
    print("Setting up project structure...")
    os.makedirs(os.path.join(PROJECT_ROOT, "data", "processed"), exist_ok=True)
    
    # Copy data splits (train/val/test)
    for split in ["train", "val", "test"]:
        src = os.path.join(DATA_SOURCE, split)
        dst = os.path.join(PROJECT_ROOT, "data", "processed", split)
        if os.path.exists(src):
            shutil.copytree(src, dst)
            print(f"  Copied {split}/")
        else:
            print(f"  WARNING: {src} not found!")
    
    # Copy metadata.csv
    meta_src = os.path.join(DATA_SOURCE, "metadata.csv")
    if os.path.exists(meta_src):
        shutil.copy2(meta_src, os.path.join(PROJECT_ROOT, "data", "processed", "metadata.csv"))
    
    # Copy source code
    src_code = os.path.join(DATA_SOURCE, "src")
    dst_code = os.path.join(PROJECT_ROOT, "src")
    if os.path.exists(src_code) and not os.path.exists(dst_code):
        shutil.copytree(src_code, dst_code)
        print("  Copied src/")
    
    print("Done!")
else:
    print("Data already set up.")

# Verify everything is in place
print(f"\nProject root: {PROJECT_ROOT}")
print(f"Data exists: {os.path.exists(os.path.join(PROJECT_ROOT, 'data', 'processed', 'train', 'input'))}")
print(f"Source code exists: {os.path.exists(os.path.join(PROJECT_ROOT, 'src', 'model', 'generator.py'))}")

## Cell 2: Install Dependencies (if needed)

These packages are usually already available on Colab runtimes, but this cell ensures nothing is missing. The `-q` flag suppresses verbose output.

In [ ]:
# Only needed if running on a fresh Colab runtime
# These are pre-installed on most Colab instances but included for safety
!pip install torch torchvision ultralytics opencv-python-headless Pillow pandas -q

## Cell 3: Add Project Root to Path

Python needs to know where our project lives so it can find the `src/` modules. Since we unzipped to `/content/project/`, we add that to `sys.path`.

In [ ]:
import sys
import os

# Add project root to Python path so we can import our modules
PROJECT_ROOT = "/content/project"
sys.path.insert(0, PROJECT_ROOT)
os.chdir(PROJECT_ROOT)

print(f"Working directory: {os.getcwd()}")
print(f"Data exists: {os.path.exists('data/processed/train/input')}")

# List what we have
for split in ["train", "val", "test"]:
    count = len(os.listdir(f"data/processed/{split}/input"))
    print(f"  {split}: {count} triplets")

## Cell 4: Import Everything

Here we pull in our custom modules from `src/model/` and standard PyTorch utilities.

- `get_dataloader` — builds a PyTorch DataLoader that feeds batches of images to the model
- `UNetGenerator` — our Generator network (U-Net architecture)
- `PatchGANDiscriminator` — our Discriminator network (classifies image patches)
- `init_weights` — initializes model weights using the Gaussian method common in GAN papers
- `config` — all the default hyperparameters defined in one place

In [ ]:
from src.model.dataset import get_dataloader
from src.model.generator import UNetGenerator, init_weights
from src.model.discriminator import PatchGANDiscriminator
from src.model import config

import torch
import torch.nn as nn
from torchvision.utils import save_image
from pathlib import Path

print("All imports successful!")

## Cell 5: Configuration

This is your control panel for the training run. Values start from `config.py` defaults, but you can override any of them here.

**Key things to know:**
- `BATCH_SIZE`: How many images are processed at once. Larger = faster training but uses more GPU memory. On a T4 (16 GB), 8 is a safe starting point.
- `LAMBDA_L1`: How much the generator is penalized for pixel-level mismatch (vs. just fooling the discriminator). The default of 100 is from the original pix2pix paper.
- `RESUME_FROM`: Set this to a checkpoint path to continue a previous training run instead of starting fresh.

In [ ]:
# ============================================================
# Override any config values here before training
# ============================================================

# Data — points to unzipped location on Colab's local disk
DATA_DIR = "data/processed"
ROOM_SIZE = 256
FURNITURE_SIZE = 128

# Training — adjust these based on your GPU memory
BATCH_SIZE = 8          # Reduce to 4 if you get OOM errors on T4
NUM_WORKERS = 2         # Colab works better with fewer workers
NUM_EPOCHS = 200
LEARNING_RATE = 0.0002
BETA1 = 0.5
BETA2 = 0.999
LAMBDA_L1 = 100         # Weight for pixel-level loss (from original pix2pix paper)
LAMBDA_GAN = 1          # Weight for adversarial (GAN) loss

# Save checkpoints and samples to Google Drive so they persist after runtime disconnects
CHECKPOINT_DIR = "/content/drive/MyDrive/693-checkpoints"
SAMPLE_DIR = "/content/drive/MyDrive/693-samples"
SAVE_EVERY = 10         # Save model every N epochs
SAMPLE_EVERY = 5        # Save sample images every N epochs

# Resume from checkpoint (set to path string to resume, or None to start fresh)
RESUME_FROM = None
# Example: RESUME_FROM = "/content/drive/MyDrive/693-checkpoints/checkpoint_epoch_0050.pt"

print("Configuration set!")
print(f"  Batch size: {BATCH_SIZE}")
print(f"  Epochs: {NUM_EPOCHS}")
print(f"  Learning rate: {LEARNING_RATE}")
print(f"  Lambda L1: {LAMBDA_L1}")
print(f"  Checkpoints → {CHECKPOINT_DIR}")
print(f"  Samples → {SAMPLE_DIR}")

## Cell 6: Create Dataloaders

A DataLoader is PyTorch's way of feeding data to the model in batches. It handles shuffling, batching, and parallel loading automatically.

- **Training loader:** Augmentation is enabled (random flips, etc.) to make the model more robust
- **Validation loader:** No augmentation — we want a consistent view of how the model is doing

The sanity check at the end confirms the data shapes are what we expect before training starts.

In [ ]:
# Create training dataloader — with augmentation and shuffling
train_loader = get_dataloader(
    DATA_DIR, "train",
    batch_size=BATCH_SIZE,
    room_size=ROOM_SIZE,
    furniture_size=FURNITURE_SIZE,
    augment=True,
    num_workers=NUM_WORKERS
)

# Create validation dataloader — no augmentation, no shuffle
val_loader = get_dataloader(
    DATA_DIR, "val",
    batch_size=BATCH_SIZE,
    room_size=ROOM_SIZE,
    furniture_size=FURNITURE_SIZE,
    augment=False,
    num_workers=NUM_WORKERS
)

print(f"Train batches per epoch: {len(train_loader)}")
print(f"Val batches: {len(val_loader)}")

# Quick sanity check — load one batch and verify shapes
batch = next(iter(train_loader))
print(f"Input shape:     {batch['input'].shape}")
print(f"Target shape:    {batch['target'].shape}")
print(f"Furniture shape: {batch['furniture'].shape}")

## Cell 7: Initialize Models

Here we create both networks, move them to the GPU, and set up the optimizers and loss functions.

**Why Adam with beta1=0.5?**
The original pix2pix paper found that reducing beta1 from the default 0.9 to 0.5 makes GAN training more stable.

**Two loss functions:**
- `BCEWithLogitsLoss` — Binary cross-entropy for the real/fake classification task ("is this a real pair or a fake one?")
- `L1Loss` — Mean absolute error at the pixel level ("how different are the generated and real images pixel by pixel?")

If `RESUME_FROM` is set, the checkpoint restores all weights and optimizer states so training continues exactly where it left off.

In [ ]:
# Select device — prefer CUDA (Nvidia GPU), then MPS (Apple), then CPU
device = torch.device("cuda" if torch.cuda.is_available() 
                      else "mps" if torch.backends.mps.is_available() 
                      else "cpu")
print(f"Using device: {device}")

# Create and initialize Generator
# U-Net uses skip connections between encoder and decoder layers,
# which helps preserve fine spatial details in the output image.
generator = UNetGenerator().to(device)
init_weights(generator)  # Gaussian weight init (mean=0, std=0.02) as in the paper
print(f"Generator parameters: {sum(p.numel() for p in generator.parameters()):,}")

# Create and initialize Discriminator
# PatchGAN classifies overlapping patches of the image rather than the whole image,
# which encourages sharper local textures.
discriminator = PatchGANDiscriminator().to(device)
init_weights(discriminator)
print(f"Discriminator parameters: {sum(p.numel() for p in discriminator.parameters()):,}")

# Optimizers — Adam with GAN-specific beta1
optimizer_G = torch.optim.Adam(generator.parameters(), lr=LEARNING_RATE, betas=(BETA1, BETA2))
optimizer_D = torch.optim.Adam(discriminator.parameters(), lr=LEARNING_RATE, betas=(BETA1, BETA2))

# Loss functions
criterion_GAN = nn.BCEWithLogitsLoss()  # For real/fake classification
criterion_L1 = nn.L1Loss()              # For pixel-level reconstruction

# Resume from checkpoint if specified
start_epoch = 0
if RESUME_FROM and os.path.exists(RESUME_FROM):
    checkpoint = torch.load(RESUME_FROM, map_location=device)
    generator.load_state_dict(checkpoint["generator"])
    discriminator.load_state_dict(checkpoint["discriminator"])
    optimizer_G.load_state_dict(checkpoint["optimizer_G"])
    optimizer_D.load_state_dict(checkpoint["optimizer_D"])
    start_epoch = checkpoint["epoch"] + 1
    print(f"Resumed from checkpoint: {RESUME_FROM} (epoch {start_epoch})")
else:
    print("Training from scratch.")

# Create output directories
Path(CHECKPOINT_DIR).mkdir(exist_ok=True)
Path(SAMPLE_DIR).mkdir(exist_ok=True)
print("Ready to train!")

## Cell 8: Helper Functions

`save_samples` generates a few images from the validation set and saves them side-by-side:

```
[ Input room ] | [ Generated ] | [ Ground truth ]
```

This is the main way to visually judge whether the model is improving epoch by epoch. Images are normalized to `[-1, 1]` during training (standard for GANs), so we undo that normalization before saving.

In [ ]:
def save_samples(generator, val_loader, device, epoch, sample_dir):
    """Generate and save sample images to visually track training progress."""
    generator.eval()  # Switch to eval mode (disables dropout, batchnorm behaves differently)
    with torch.no_grad():  # Don't compute gradients — we're just doing inference
        batch = next(iter(val_loader))
        inp = batch["input"].to(device)
        furn = batch["furniture"].to(device)
        target = batch["target"].to(device)
        
        # Run the generator forward pass to produce fake furnished rooms
        fake = generator(inp, furn)
        
        # Denormalize from [-1,1] to [0,1] for saving
        # During training images are normalized: pixel = (pixel - 0.5) / 0.5
        # To reverse: pixel = pixel * 0.5 + 0.5
        inp_denorm = inp * 0.5 + 0.5
        fake_denorm = fake * 0.5 + 0.5
        target_denorm = target * 0.5 + 0.5
        
        # Stack side by side: input | generated | target (first 4 images in batch)
        n = min(4, inp.size(0))
        comparison = torch.cat([inp_denorm[:n], fake_denorm[:n], target_denorm[:n]], dim=3)
        
        save_path = os.path.join(sample_dir, f"epoch_{epoch:04d}.png")
        save_image(comparison, save_path, nrow=1, padding=2)
        print(f"  Samples saved to {save_path}")
    
    generator.train()  # Switch back to train mode

## Cell 9: Training Loop

This is the heart of GAN training. Each epoch iterates over all batches and performs two alternating updates:

**Step 1 — Train Discriminator (D):**
- Feed a real (input, target) pair → D should output 1 ("real")
- Feed a fake (input, generated) pair → D should output 0 ("fake")
- Update D's weights to improve at this classification

**Step 2 — Train Generator (G):**
- Feed the same fake pair to D — but now G wants D to output 1 ("fool" D)
- Also penalize G for being far from the real target pixel-by-pixel (L1 loss)
- Update G's weights to fool D AND match the target

The key trick: `.detach()` in Step 1 prevents the discriminator's backward pass from updating the generator. Each network only updates based on its own loss.

In [ ]:
# ============================================================
# MAIN TRAINING LOOP
# ============================================================

print("=" * 60)
print("Starting training")
print("=" * 60)

# Track losses for plotting later
history = {"d_loss": [], "g_loss": [], "g_gan": [], "g_l1": []}

for epoch in range(start_epoch, NUM_EPOCHS):
    generator.train()
    discriminator.train()
    
    # Running totals for this epoch
    epoch_d_loss = 0.0
    epoch_g_loss = 0.0
    epoch_g_gan = 0.0
    epoch_g_l1 = 0.0
    num_batches = 0
    
    for batch_idx, batch in enumerate(train_loader):
        # Move data to GPU
        real_input = batch["input"].to(device)        # Room without bed
        real_target = batch["target"].to(device)       # Room with bed (ground truth)
        furniture = batch["furniture"].to(device)      # Cropped bed image
        
        # -------------------------------------------------------
        # 1) Train Discriminator
        # -------------------------------------------------------
        # The discriminator learns to tell real from fake.
        
        # Generate a fake furnished room
        fake_target = generator(real_input, furniture)
        
        # D judges the real pair (should say "real" → label = 1)
        pred_real = discriminator(real_input, real_target)
        real_labels = torch.ones_like(pred_real)
        loss_D_real = criterion_GAN(pred_real, real_labels)
        
        # D judges the fake pair (should say "fake" → label = 0)
        # .detach() prevents gradients from flowing back into G during D's update
        pred_fake = discriminator(real_input, fake_target.detach())
        fake_labels = torch.zeros_like(pred_fake)
        loss_D_fake = criterion_GAN(pred_fake, fake_labels)
        
        # Total D loss (average of real and fake)
        loss_D = 0.5 * (loss_D_real + loss_D_fake)
        
        optimizer_D.zero_grad()  # Clear old gradients
        loss_D.backward()        # Compute new gradients
        optimizer_D.step()       # Update D's weights
        
        # -------------------------------------------------------
        # 2) Train Generator
        # -------------------------------------------------------
        # The generator learns to fool D AND match the real target.
        
        # D judges the fake (G wants D to say "real" → label = 1)
        # Note: we do NOT detach here — gradients need to flow back into G
        pred_fake_for_G = discriminator(real_input, fake_target)
        loss_G_GAN = criterion_GAN(pred_fake_for_G, torch.ones_like(pred_fake_for_G))
        
        # Pixel-level loss — how close is the fake to the real target?
        loss_G_L1 = criterion_L1(fake_target, real_target)
        
        # Total G loss: GAN loss + weighted L1 loss
        # LAMBDA_L1=100 means pixel accuracy is weighted heavily relative to fooling D
        loss_G = LAMBDA_GAN * loss_G_GAN + LAMBDA_L1 * loss_G_L1
        
        optimizer_G.zero_grad()  # Clear old gradients
        loss_G.backward()        # Compute new gradients
        optimizer_G.step()       # Update G's weights
        
        # Accumulate losses for logging
        epoch_d_loss += loss_D.item()
        epoch_g_loss += loss_G.item()
        epoch_g_gan += loss_G_GAN.item()
        epoch_g_l1 += loss_G_L1.item()
        num_batches += 1
    
    # --- End of epoch: compute averages ---
    avg_d = epoch_d_loss / num_batches
    avg_g = epoch_g_loss / num_batches
    avg_g_gan = epoch_g_gan / num_batches
    avg_g_l1 = epoch_g_l1 / num_batches
    
    history["d_loss"].append(avg_d)
    history["g_loss"].append(avg_g)
    history["g_gan"].append(avg_g_gan)
    history["g_l1"].append(avg_g_l1)
    
    print(f"Epoch [{epoch+1}/{NUM_EPOCHS}] "
          f"D_loss: {avg_d:.4f} | G_loss: {avg_g:.4f} "
          f"(GAN: {avg_g_gan:.4f}, L1: {avg_g_l1:.4f})")
    
    # Save sample images periodically
    if (epoch + 1) % SAMPLE_EVERY == 0:
        save_samples(generator, val_loader, device, epoch + 1, SAMPLE_DIR)
    
    # Save checkpoint periodically
    if (epoch + 1) % SAVE_EVERY == 0:
        ckpt_path = os.path.join(CHECKPOINT_DIR, f"checkpoint_epoch_{epoch+1:04d}.pt")
        torch.save({
            "generator": generator.state_dict(),
            "discriminator": discriminator.state_dict(),
            "optimizer_G": optimizer_G.state_dict(),
            "optimizer_D": optimizer_D.state_dict(),
            "epoch": epoch,
        }, ckpt_path)
        print(f"  Checkpoint saved to {ckpt_path}")

# Save final model after all epochs complete
final_path = os.path.join(CHECKPOINT_DIR, "checkpoint_final.pt")
torch.save({
    "generator": generator.state_dict(),
    "discriminator": discriminator.state_dict(),
    "optimizer_G": optimizer_G.state_dict(),
    "optimizer_D": optimizer_D.state_dict(),
    "epoch": NUM_EPOCHS - 1,
}, final_path)
print(f"\nTraining complete! Final checkpoint: {final_path}")

## Cell 10: Plot Training Losses

After training, visualizing the loss curves helps diagnose how training went.

**What to look for:**
- **D loss** should settle near ~0.5 (discriminator can't reliably tell real from fake — that's good!)
- **G GAN loss** should decrease as the generator gets better at fooling D
- **G L1 loss** should decrease as the generator's output gets closer to the ground truth
- If D loss collapses to 0 early, the discriminator is winning too easily — the generator needs more capacity or a lower learning rate
- If G loss collapses, the generator may have found a "shortcut" (mode collapse)

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Plot D loss
axes[0].plot(history["d_loss"], label="D loss")
axes[0].set_title("Discriminator Loss")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].legend()
axes[0].grid(True)

# Plot G total loss
axes[1].plot(history["g_loss"], label="G total loss", color="orange")
axes[1].set_title("Generator Total Loss")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Loss")
axes[1].legend()
axes[1].grid(True)

# Plot G components (GAN vs L1) separately to see each contribution
axes[2].plot(history["g_gan"], label="G GAN loss", color="red")
axes[2].plot(history["g_l1"], label="G L1 loss", color="green")
axes[2].set_title("Generator Loss Components")
axes[2].set_xlabel("Epoch")
axes[2].set_ylabel("Loss")
axes[2].legend()
axes[2].grid(True)

plt.tight_layout()
plt.savefig(os.path.join(SAMPLE_DIR, "training_losses.png"), dpi=150)
plt.show()
print("Loss plot saved!")

## Cell 11: View Sample Results

Here we display the most recent sample image saved during training.

Each row shows three images for one validation example:
- **Left:** Input room (no bed)
- **Middle:** What the generator produced
- **Right:** The actual ground truth (what we want the generator to produce)

As training progresses, the middle column should look increasingly similar to the right column.

In [ ]:
from IPython.display import Image, display
import glob

# Show the most recent sample image
sample_files = sorted(glob.glob(os.path.join(SAMPLE_DIR, "epoch_*.png")))
if sample_files:
    latest = sample_files[-1]
    print(f"Latest sample: {latest}")
    print("Left: Input (room without bed) | Middle: Generated | Right: Ground truth")
    display(Image(filename=latest, width=900))
else:
    print("No sample images found yet. Train for at least SAMPLE_EVERY epochs.")